# RoboBarista Vision — FAIRINO FR5 und 3 RGB-D-Kameras

**Phase 1: sehen, wo die Dinge wirklich sind.** Die Szene wird aus einem bestehenden MIT-lizenzierten FR5-Modell aufgebaut. Die drei Kameras erkennen ArUco-Marker und schätzen 6D-Posen. Die **echte MuJoCo-Pose wird nur für die anschließende Fehlerauswertung** verwendet.

**Wichtig:** Die Geometrietests wurden lokal bestanden. Ein erfolgreicher vollständiger Render- und Genauigkeitsdurchlauf ist noch nicht nachgewiesen; **5 mm / 5° sind Testziele**, keine zugesicherte Genauigkeit.

**Keine GPU erforderlich:** Colab kann mit Hardwarebeschleuniger **Keine** gestartet werden. Schritt 2 installiert automatisch die nötigen OSMesa-Systembibliotheken. Mit GPU wird alternativ EGL verwendet.

Du brauchst keine Dateien hochzuladen. In Colab **Laufzeit → Alle ausführen**. 


In [ ]:
# 1. GitHub-Projekt laden (keine ZIP nötig)
import os, subprocess, sys, pathlib
repo = pathlib.Path('/content/robobarista-vision')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Coffee-Tobi/robocasa.git', str(repo)], check=True)
else:
    # Bei erneutem Ausführen aktuelle Verbesserungen von GitHub holen.
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
print('Projekt:', repo)


In [ ]:
# 2. Systemgrafik (CPU/OSMesa) und Python-Pakete installieren
# Ohne GPU benötigt MuJoCo echte OSMesa-Systembibliotheken; pip allein reicht nicht.
import os, subprocess, sys, ctypes.util
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'libosmesa6-dev',
                'libgl1', 'libglx-mesa0', 'libgl1-mesa-dri'], check=True)
osmesa_lib = ctypes.util.find_library('OSMesa')
print('OSMesa-Systembibliothek:', osmesa_lib)
if not osmesa_lib:
    raise RuntimeError('OSMesa fehlt trotz apt-Installation. Bitte die Ausgabe von Schritt 2 teilen.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'requirements.txt'], check=True)

# Eigenständiger Importcheck im richtigen Rendering-Backend (ohne GPU: OSMesa).
backend = 'egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
check_env = os.environ.copy()
check_env['MUJOCO_GL'] = backend
if backend == 'osmesa':
    check_env['PYOPENGL_PLATFORM'] = 'osmesa'
print('MuJoCo-Importtest mit Backend:', backend, flush=True)
subprocess.run([sys.executable, '-c',
                'import mujoco; print("MuJoCo:", mujoco.__version__, "Import OK")'],
               env=check_env, check=True)
import cv2, numpy as np
print('OpenCV:', cv2.__version__, 'NumPy:', np.__version__)


In [ ]:
# 3. Schnelle Tests der Kamera-Mathematik — unabhängig von MuJoCo
import subprocess,sys
subprocess.run([sys.executable,'-m','pytest','-q','tests'],check=True)


In [ ]:
# 4. Den FAIRINO FR5 und drei RGB-D-Kameras virtuell rendern
# Die korrekte Backend-Wahl erfolgt vor dem Import von MuJoCo im subprocess.
import os, subprocess, sys
settings = os.environ.copy()
settings["MUJOCO_GL"] = "egl" if os.path.exists("/dev/nvidia0") else "osmesa"
if settings["MUJOCO_GL"] == "osmesa":
    settings["PYOPENGL_PLATFORM"] = "osmesa"
cmd = [sys.executable, "-u", "-m", "robobarista.demo",
       "--trials", "3", "--width", "960", "--height", "720",
       "--min-cameras", "2", "--max-translation-mm", "5",
       "--max-rotation-deg", "5"]
print("Rendering-Backend:", settings["MUJOCO_GL"], flush=True)
run = subprocess.run(cmd, env=settings, cwd=str(repo),
                     stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                     text=True, errors="replace", check=False)
print("Benchmark Exit-Code:", run.returncode, flush=True)
print("\n=== PROGRAMMAUSGABE (stdout) ===")
print(run.stdout[-20000:] if run.stdout.strip() else "(leer)")
print("\n=== FEHLERMELDUNGEN (stderr) ===")
print(run.stderr[-20000:] if run.stderr.strip() else "(leer)")
if run.returncode != 0:
    print("\nWICHTIG: Noch kein bestandener Genauigkeitstest.")
    print("Bitte kopiere bei einem Fehler den Abschnitt FEHLERMELDUNGEN (stderr) hierher.")
else:
    print("\nBenchmark erfolgreich abgeschlossen.")


In [ ]:
# 5. Kameras und Tiefenbilder nebeneinander ansehen
from IPython.display import display,Image
from pathlib import Path
out=Path('runs/first_benchmark')
for filename in ['rgbd_montage.jpg','pose_comparison.png']:
    f=out/filename
    if f.exists():
        print(filename)
        display(Image(filename=str(f)))
    else:
        print('Nicht vorhanden:',f)


In [ ]:
# 6. Genaue Abweichungen ansehen und Bilder/CSV bei Bedarf herunterladen
import json,pandas as pd
p=out/'summary.json'
if p.exists():
    report=json.loads(p.read_text())
    print(json.dumps(report,indent=2,ensure_ascii=False))
    error_csv=out/'pose_errors.csv'
    if error_csv.exists(): display(pd.read_csv(error_csv))
else:
    print('Kein Messbericht. Bitte Fehlermeldung aus Zelle 4 schicken.')


### Was danach kommt

Erst wenn Kamerasicht, Rekonstruktionsfehler und die Zuordnung der Tag-Achsen stimmen, beginnen wir mit einer **bewegten Kännchen-Demonstration**. Wir bauen keine eigene Milchphysik und trainieren jetzt noch kein VLA-Modell.

[Code und Roadmap auf GitHub](https://github.com/Coffee-Tobi/robocasa)